In [ ]:
# This cell installs the libraries on your local machine via the Colab interface
!pip install ultralytics roboflow --upgrade

In [ ]:
from roboflow import Roboflow
import os

!pip install roboflow --upgrade

from roboflow import Roboflow
rf = Roboflow(api_key="5S2rL8AmmprxjfrlMN3z")
project = rf.workspace("counting-6v260").project("optimized_bikeparking-duwzk")
version = project.version(1)
dataset = version.download("yolov8")

# This path will now point to a location on your local H: drive
data_yaml_path = os.path.join(dataset.location, 'data.yaml')
print(f"Dataset downloaded to: {dataset.location}")

In [ ]:
!nvidia-smi

Sat Oct 18 19:05:06 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.15                 Driver Version: 581.15         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090      WDDM  |   00000000:01:00.0  On |                  N/A |
| 30%   37C    P8             34W /  370W |    1168MiB /  24576MiB |      9%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# --- Step 1: Import os and set the environment variable ---
# This is the definitive fix for the "OMP: Error #15" crash.
# It MUST be run before importing torch or ultralytics.
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

# --- Step 2: Import remaining libraries ---
from ultralytics import YOLO
import torch

# --- Step 3: Run Training inside a main guard ---
# This is a best practice for PyTorch on Windows to ensure stability.
if __name__ == '__main__':

    # Optional: Clear GPU cache from any previous failed runs.
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Load the pre-trained YOLO26 Large model.
    # Available sizes: yolo26n.pt, yolo26s.pt, yolo26m.pt, yolo26l.pt, yolo26x.pt
    model = YOLO('yolo26l.pt')

    # Train the model with all stability fixes applied.
    results = model.train(
        data=data_yaml_path,  # This variable must be defined by running your Roboflow cell first.
        epochs=250,
        patience=50,

        # Using a conservative batch size to guarantee stability.
        batch=16,

        imgsz=640,
        name='yolo26l_bike_run',

        # Essential for preventing DataLoader errors on Windows.
        workers=0,
    )

    print("\n✅ Training finished successfully!")
    print(f"✅ Model and results have been saved to your local directory: {results.save_dir}")